### Gold Helpers
This notebook has a reusable function called `write_to_gold()` that all gold notebooks use to save their data.

---

#### What does `write_to_gold()` do?

It handles **two situations**:

**1. First time run (table doesn't exist yet)**
- Creates the gold table and writes all the data into it
- Adds `created_timestamp` and `updated_timestamp` columns automatically

**2. Table already exists (incremental load)**
- Uses Delta Lake **MERGE** to compare new data with existing data
- If a row already exists (matched by merge condition) → **updates** listed columns
- If a row is brand new (not matched) → **inserts** it

---

#### Parameters it takes:

| Parameter | What it means |
| --- | --- |
| `input_df` | The final DataFrame you want to save |
| `target_table` | Full table name like `formula1_incr.gold.dim_races` |
| `merge_condition` | How to match rows, e.g. `t.season = s.season AND t.round = s.round` |
| `columns_to_update` | List of columns to refresh when a match is found |

---

#### How is this different from `write_to_silver()`?

| | `write_to_silver()` | `write_to_gold()` |
| --- | --- | --- |
| Update condition | Only updates if `s.batch_id >= t.batch_id` | Always updates matched rows |
| Use case | Incremental raw data loads | Derived/joined tables that should always reflect latest |

---

#### Why we use this helper:
- Avoids repeating merge code in every gold notebook
- All gold tables follow the same pattern
- Handles both first-run and re-run automatically

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

In [0]:
def write_to_gold(
    input_df,
    target_table,
    merge_condition,
    columns_to_update
):
    
    final_df = (
        input_df
        .withColumn('created_timestamp', current_timestamp())
        .withColumn('updated_timestamp', current_timestamp())
    )
    if not spark.catalog.tableExists(target_table):
        (
            final_df
            .write
            .mode("overwrite")
            .format("delta")
            .saveAsTable(target_table)
        )
    else:
        delta_table = DeltaTable.forName(spark, target_table)
        update_map = {column: f's.{column}' for column in columns_to_update}
        update_map['updated_timestamp'] = 's.updated_timestamp'
        (
            delta_table.alias('t')
            .merge(
                final_df.alias('s'),
                merge_condition
            )
            .whenMatchedUpdate(
                set = update_map
            )
            .whenNotMatchedInsertAll()
            .execute()
        )